# CMI with $S$

Calculating CMI with density matrices 

\begin{equation}

CMI = I(A:C|B) = S(AB) + S(BC) - S(ABC) - S(B),

\end{equation}

where $S(Q)$ is the von Neumann entropy of the $Q$ subsystem:

\begin{equation}

S(Q) = - tr ( \rho_{Q} \log \rho_Q ),

\end{equation}

with

\begin{equation}

\rho_{Q} = tr_{\bar{Q}} (\rho).

\end{equation}

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
import math
from functools import reduce

We will start from defining the GHZ state:

\begin{equation}

\ket{\text{GHZ}}= \frac{1}{\sqrt{2}}(\ket{0}^{\otimes N}+ \ket{1}^{\otimes N})

\end{equation}

and the density matrix:

\begin{equation}

\rho_0 = \ket{\text{GHZ}} \bra{\text{GHZ}}

\end{equation}

In [30]:
#Defining rho_0

# -------------------------
# Tensor product helper
# -------------------------
def kron_power(v, N):
    """Compute v ⊗ v ⊗ ... ⊗ v (N times)"""
    return reduce(np.kron, [v] * N)

# -------------------------
# Parameters
# -------------------------
N = 12  # number of qubits

# -------------------------
# Basis states |0> and |1>
# -------------------------
ket_0 = np.array([1, 0])
ket_1 = np.array([0, 1])

# -------------------------
# Build product states
# |0...0> and |1...1>
# -------------------------
ket_0N = kron_power(ket_0, N)
ket_1N = kron_power(ket_1, N)

# -------------------------
# GHZ state
# -------------------------
ket_GHZ = (ket_0N + ket_1N) / np.sqrt(2)

print("GHZ state shape:", ket_GHZ.shape)

# -------------------------
# Density matrix
# -------------------------
rho_GHZ = np.outer(ket_GHZ, ket_GHZ.conj())

print("Density matrix shape:", rho_GHZ.shape)


GHZ state shape: (4096,)
Density matrix shape: (4096, 4096)


Subsequantly, we define the noise channel:

\begin{equation}

\mathcal{E}^i_p(\cdot) = (1-p)(\cdot) + pX_i(\cdot)X_i

\end{equation}

acting on every qubit.

In [31]:
#Defining the noise channel

#defining Pauli X matrix 

Pauli_X= np.array([[0, 1],[1, 0]])
print(Pauli_X)

#defining identity matrix 

I = np.eye(2)


def local_operator(op, i, N):
    """
    Build operator acting on i-th qubit (0-based index) in N-qubit system.
    """
    ops = [I] * N
    ops[i] = op
    return reduce(np.kron, ops)

def bit_flip_channel(rho, p, i, N):
    X_i = local_operator(Pauli_X, i, N)
    return (1 - p) * rho + p * (X_i @ rho @ X_i)

#Defining repeated tensor product
def kron_power(A, d):
    """ 
    Function computing tensor product of matrix A repreted d times
    """
    if d == 0:
        return np.array([[1]])  # identity for tensor product
    return reduce(np.kron, [A] * d)


#Noise channel for 1 qubit

def noise_channel_i(i,p,N,rho):
    """ 
    Function implementing the application of the noise channel on qubit i. 
    Returns the matrix rho_p_i after the application of the noise on one qubit
    i - qubit number
    p - noise rate 
    N - total system size
    rho - total density matrix
    """
    left = kron_power(I,i-1)
    right = kron_power(I,N-i)
    
    
    matrices = [left, Pauli_X, right]
    X_i = reduce(np.kron, matrices)

    print(rho.shape)
    print(X_i.shape)


    rho_p_i = (1-p)*rho + p* (X_i @ rho @ X_i)
    return rho_p_i

[[0 1]
 [1 0]]


In [33]:
#Test 

# apply channel on qubit 0
p = 0.1
rho_out = bit_flip_channel(rho_GHZ, p, i=0, N=N)

print(rho_out)

[[0.45 0.   0.   ... 0.   0.   0.45]
 [0.   0.   0.   ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.   0.  ]
 ...
 [0.   0.   0.   ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.   0.  ]
 [0.45 0.   0.   ... 0.   0.   0.45]]


In [20]:
#Test 

M=3
left_1 = kron_power(I,2-1)
left_2 = kron_power(I,M-2)
print(left_1)
print(left_2)
matrices = [left_1, Pauli_X, left_2]

#result = np.kron(np.kron(left_1, Pauli_X), left_2)
result = reduce(np.kron, matrices)
print(result)

[[1 0]
 [0 1]]
[[1 0]
 [0 1]]
[[0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 0]
 [1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 1]
 [0 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0]]


Now, we will define functions to caluclate von Neuman entropies of the whole matrix and the reduced density matrices to subsequently calculate CMI. 